# 🏥 Patient Pathway Analysis - Database Initialisation

**Objective:** Load raw Synthea CSV files, build a normalised SQLite relational database, and verify data integrity before analysis.

**Data source:** [Synthea](https://synthea.mitre.org/) — open-source synthetic patient data  
**Stack:** Python · pandas · SQLite  
**Output:** `../database/patient_journey.db`

---
**Schema overview:**
- `patients` — demographic profile (primary key: `patient_id`)
- `encounters` — healthcare visits (FK → `patient_id`)
- `conditions` — diagnoses (FK → `patient_id`)
- `procedures` — medical acts performed (FK → `patient_id`)

# 1. Imports & Environment Check

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os

print("Working directory:", os.getcwd())
print("Raw data files available:", os.listdir("../data/raw"))

Working directory: F:\BD\patient-pathway-analysis\notebooks
Raw data files available: ['conditions.csv', 'encounters.csv', 'patients.csv', 'procedures.csv']


# 2. Data Ingestion & Initial Shape Inspection

We start our pipeline by loading the raw clinical datasets from our local storage (`../data/raw/`) into Pandas DataFrames. 
During this process, we immediately standardize the database schemas by converting all column headers to lowercase and replacing spaces with underscores. This initial engineering step prevents syntax errors during future SQL queries and ensures uniform features across datasets.

In [10]:
patients   = pd.read_csv("../data/raw/patients.csv")
encounters = pd.read_csv("../data/raw/encounters.csv")
conditions = pd.read_csv("../data/raw/conditions.csv")
procedures = pd.read_csv("../data/raw/procedures.csv")

for df in [patients, encounters, conditions, procedures]:
    df.columns = df.columns.str.lower().str.replace(' ', '_')

print(f"patients   : {patients.shape[0]:,} rows, {patients.shape[1]} columns")
print(f"encounters : {encounters.shape[0]:,} rows, {encounters.shape[1]} columns")
print(f"conditions : {conditions.shape[0]:,} rows, {conditions.shape[1]} columns")
print(f"procedures : {procedures.shape[0]:,} rows, {procedures.shape[1]} columns")

patients   : 1,146 rows, 28 columns
encounters : 67,164 rows, 15 columns
conditions : 41,295 rows, 7 columns
procedures : 186,160 rows, 10 columns


In [8]:
# Quick preview of each raw file
for name, df in [("patients", patients), ("encounters", encounters),
                 ("conditions", conditions), ("procedures", procedures)]:
    print(f"\n--- {name} (columns) ---")
    print(list(df.columns))


--- patients (columns) ---
['id', 'birthdate', 'deathdate', 'ssn', 'drivers', 'passport', 'prefix', 'first', 'middle', 'last', 'suffix', 'maiden', 'marital', 'race', 'ethnicity', 'gender', 'birthplace', 'address', 'city', 'state', 'county', 'fips', 'zip', 'lat', 'lon', 'healthcare_expenses', 'healthcare_coverage', 'income']

--- encounters (columns) ---
['id', 'start', 'stop', 'patient', 'organization', 'provider', 'payer', 'encounterclass', 'code', 'description', 'base_encounter_cost', 'total_claim_cost', 'payer_coverage', 'reasoncode', 'reasondescription']

--- conditions (columns) ---
['start', 'stop', 'patient', 'encounter', 'system', 'code', 'description']

--- procedures (columns) ---
['start', 'stop', 'patient', 'encounter', 'system', 'code', 'description', 'base_cost', 'reasoncode', 'reasondescription']


## 3. Build the SQLite Relational Database

We create a normalised schema with foreign key constraints to enforce referential integrity across tables.

In [4]:
conn = sqlite3.connect("../database/patient_journey.db")
cur  = conn.cursor()

cur.execute("PRAGMA foreign_keys = ON")

cur.executescript("""
CREATE TABLE IF NOT EXISTS patients (
    patient_id  TEXT PRIMARY KEY,
    gender      TEXT CHECK(gender IN ('M', 'F')),
    birth_date  DATE NOT NULL,
    city        TEXT,
    deceased    INTEGER DEFAULT 0 CHECK(deceased IN (0, 1)),
    death_date  DATE
);

CREATE TABLE IF NOT EXISTS encounters (
    encounter_id    TEXT PRIMARY KEY,
    patient_id      TEXT NOT NULL,
    encounter_date  DATE,
    encounter_type  TEXT,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
);

CREATE TABLE IF NOT EXISTS conditions (
    condition_id    TEXT PRIMARY KEY,
    patient_id      TEXT NOT NULL,
    condition_name  TEXT,
    start_date      DATE,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
);

CREATE TABLE IF NOT EXISTS procedures (
    procedure_id    TEXT PRIMARY KEY,
    patient_id      TEXT NOT NULL,
    procedure_name  TEXT,
    procedure_date  DATE,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
);
""")

conn.commit()
print("✅ Schema created successfully.")

✅ Schema created successfully.


## 4. Clean & Inject Data

We select and rename only the relevant columns from each raw CSV before loading into SQLite.

In [5]:
# --- PATIENTS ---
patients_clean = (
    patients[['Id', 'GENDER', 'BIRTHDATE', 'CITY', 'DEATHDATE']]
    .rename(columns={
        'Id':        'patient_id',
        'GENDER':    'gender',
        'BIRTHDATE': 'birth_date',
        'CITY':      'city',
        'DEATHDATE': 'death_date'
    })
    .assign(deceased=lambda df: df['death_date'].notna().astype(int))
)
patients_clean.to_sql('patients', conn, if_exists='replace', index=False)
print(f"✅ patients   : {len(patients_clean):,} rows loaded")

# --- ENCOUNTERS ---
encounters_clean = (
    encounters[['Id', 'PATIENT', 'START', 'ENCOUNTERCLASS']]
    .rename(columns={
        'Id':            'encounter_id',
        'PATIENT':       'patient_id',
        'START':         'encounter_date',
        'ENCOUNTERCLASS':'encounter_type'
    })
)
encounters_clean.to_sql('encounters', conn, if_exists='replace', index=False)
print(f"✅ encounters : {len(encounters_clean):,} rows loaded")

# --- CONDITIONS ---
conditions_clean = (
    conditions[['CODE', 'PATIENT', 'DESCRIPTION', 'START']]
    .rename(columns={
        'CODE':        'condition_id',
        'PATIENT':     'patient_id',
        'DESCRIPTION': 'condition_name',
        'START':       'start_date'
    })
)
conditions_clean.to_sql('conditions', conn, if_exists='replace', index=False)
print(f"✅ conditions : {len(conditions_clean):,} rows loaded")

# --- PROCEDURES ---
procedures_clean = (
    procedures[['START', 'PATIENT', 'ENCOUNTER', 'DESCRIPTION']]
    .rename(columns={
        'START':       'procedure_date',
        'PATIENT':     'patient_id',
        'ENCOUNTER':   'encounter_id',
        'DESCRIPTION': 'procedure_name'
    })
)
procedures_clean.to_sql('procedures', conn, if_exists='replace', index=False)
print(f"✅ procedures : {len(procedures_clean):,} rows loaded")

conn.commit()

✅ patients   : 1,146 rows loaded
✅ encounters : 67,164 rows loaded
✅ conditions : 41,295 rows loaded
✅ procedures : 186,160 rows loaded


## 5. Verify Data Integrity

In [6]:
# Row counts per table
tables = ['patients', 'encounters', 'conditions', 'procedures']

print("=== Row counts ===")
for table in tables:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]
    print(f"  {table:<15}: {n:,} rows")

# Sample preview
print("\n=== Sample rows ===")
for table in tables:
    print(f"\n--- {table} ---")
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 3", conn))

# Null check on patients
print("\n=== Null values in patients ===")
df_patients = pd.read_sql("SELECT * FROM patients", conn)
print(df_patients.isnull().sum())

=== Row counts ===
  patients       : 1,146 rows
  encounters     : 67,164 rows
  conditions     : 41,295 rows
  procedures     : 186,160 rows

=== Sample rows ===

--- patients ---


,patient_id,gender,birth_date,city,death_date,deceased
0,ca73401b-6e45-ae91-4a1d-04b9ccc7fcb6,M,1992-03-06,Boston,None,0
1,5939ede0-f907-8988-0403-fe4156a20af5,M,1978-12-29,Ludlow,None,0
2,0b8736b7-6317-9207-f29a-77a96272dcca,F,2020-08-30,Wilmington,None,0



--- encounters ---


,encounter_id,patient_id,encounter_date,encounter_type
0,ca73401b-6e45-ae91-7f20-878335c13ea2,ca73401b-6e45-ae91-4a1d-04b9ccc7fcb6,2009-04-24T03:54:19Z,wellness
1,5939ede0-f907-8988-f5f5-3b5de078b634,5939ede0-f907-8988-0403-fe4156a20af5,1997-02-21T08:47:33Z,wellness
2,0b8736b7-6317-9207-e99d-30eb50adf573,0b8736b7-6317-9207-f29a-77a96272dcca,2020-08-30T01:51:06Z,wellness



--- conditions ---


,condition_id,patient_id,condition_name,start_date
0,224299000,5939ede0-f907-8988-0403-fe4156a20af5,Received higher education (finding),1997-02-21
1,314529007,ef6331b4-9a11-2a21-1c7d-33d8e6514514,Medication review due (situation),2016-05-07
2,160968000,ca73401b-6e45-ae91-4a1d-04b9ccc7fcb6,Risk activity involvement (finding),2009-04-24



--- procedures ---


,procedure_date,patient_id,encounter_id,procedure_name
0,2016-06-23T03:16:23Z,74de14d6-55c5-c1db-17a2-63231cf71cd7,74de14d6-55c5-c1db-ac7a-21bbf7398c4e,Throat culture (procedure)
1,2016-11-05T13:18:59Z,ef6331b4-9a11-2a21-1c7d-33d8e6514514,ef6331b4-9a11-2a21-c621-763bf9eade94,Medication reconciliation (procedure)
2,2017-05-12T03:54:19Z,ca73401b-6e45-ae91-4a1d-04b9ccc7fcb6,ca73401b-6e45-ae91-6af7-ee2f75b8dafc,Assessment of health and social care needs (pr...



=== Null values in patients ===
patient_id       0
gender           0
birth_date       0
city             0
death_date    1000
deceased         0
dtype: int64


## Summary

The relational database has been successfully built from raw Synthea CSV files.

- Four tables with primary keys and foreign key constraints
- Data cleaned and columns renamed for consistency
- Referential integrity verified

**Next step:** `02_sql_analysis.ipynb`- descriptive analysis and temporal pathway reconstruction using SQL.